# Simplified Downhole Collection

This notebook demonstrates how to create a **`DownholeCollection`** geoscience object using the **typed objects API** (`evo.objects.typed`).

A downhole collection describes a set of drill holes via:
- a **path** table (the geometry of each hole as `distance`/`azimuth`/`dip` steps),
- a **holes** table that slices the path into individual holes (`hole_index`/`offset`/`count`),
- a **properties** table with the collar location and depths per hole (`hole_id`, `x`, `y`, `z`, `final`, `target`, `current`),
- zero or more **collections** of downhole measurements (here, a `DistanceCollection`).

## Requirements

You must have a Seequent account with the Evo entitlement. You'll need:
- The client ID of your Evo application
- The callback/redirect URL of your Evo application

To obtain these credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens).

## 1. Authentication

First, we authenticate using the `ServiceManagerWidget`. This handles OAuth login and workspace selection.

In [ ]:
from evo.notebooks import ServiceManagerWidget

# Replace with your Evo app credentials
client_id = "<your-client-id>"
redirect_url = "<your-redirect-url>"

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id,
    redirect_url=redirect_url,
    cache_location="./notebook-data",
).login()

## 2. Load the Widgets Extension

The `evo.widgets` extension enables rich HTML rendering of Evo objects in Jupyter notebooks. After loading, objects will automatically display with formatted tables and clickable links.

In [ ]:
%load_ext evo.widgets

## 3. Build the Downhole Collection

This creates a downhole collection with two holes. See `DownholeCollectionData` to understand what all the data fields represent.

In [ ]:
from datetime import date, datetime, timezone

import numpy as np
import pandas as pd

from evo.objects.typed.downhole_collection import DistanceCollection, DownholeCollectionData

# Concatenated path table representing two holes: hole 0 has 4 rows, hole 1 has 3 rows. It has one custom attribute "attr".
path = pd.DataFrame(
    {
        "distance": [0.0, 10.0, 20.0, 30.0, 0.0, 15.0, 30.0],
        "azimuth": [0.0, 0.0, 0.0, 0.0, 90.0, 90.0, 90.0],
        "dip": [90.0, 90.0, 90.0, 90.0, 45.0, 45.0, 45.0],
        "attr": ["a", "b", "a", "c", "c", "c", "c"],
    }
)

# Map each hole to a contiguous slice of the path table. There are two rows, which means there are two holes.
# Note the `astype`. The columns must have those exact types.
holes = pd.DataFrame(
    {
        "hole_index": [0, 1],
        "offset": [0, 4],
        "count": [4, 3],
    }
).astype({"hole_index": np.int32, "offset": np.uint64, "count": np.uint64})

# There are two custom attributes for each hole. This is optional.
attributes = pd.DataFrame(
    {
        "attr_str": ["a", "b"],
        "attr_int": [1, 2],
    }
)

# These are mandatory per-hole properties.
properties = pd.DataFrame(
    {
        "hole_id": ["H001", "H002"],
        "x": [100.0, 200.0],
        "y": [150.0, 300.0],
        "z": [0.0, 50.0],
        "final": [30.0, 30.0],
        "target": [25.0, 25.0],
        "current": [30.0, 28.0],
    }
)

# This represents a collection of surveyed data, e.g. depth measurements.
# Note the `astype` for the `holes` argument. The columns must have those exact types.
# This collection has three custom attributes, in addition to the depths.
collection = DistanceCollection(
    name="collection1",
    collection_type="distance",
    holes=pd.DataFrame(
        {
            "hole_index": [0],
            "offset": [0],
            "count": [4],
        }
    ).astype({"hole_index": np.int32, "offset": np.uint64, "count": np.uint64}),
    distance_table=pd.DataFrame(
        {
            "distance": [0.0, 10.0, 20.0, 30.0],
            "attr_str": ["a", "b", "a", "c"],
            "attr_dt": [date(2000, 1, 1), date(2000, 1, 2), date(2000, 1, 3), date(2000, 1, 4)],
            "attr_num": [1.1, 2.2, 3.3, 4.4],
        }
    ),
)

dhc_data = DownholeCollectionData(
    name=f"Example DHC {datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')}",
    description="A downhole collection created with the typed objects API",
    tags={"site": "alpha", "status": "active"},
    path=path,
    holes=holes,
    properties=properties,
    attributes=attributes,
    collections=[collection],
    distance_unit="m",
    desurvey="trench",
)

## 4. Create the Object in Evo

`DownholeCollection.create` uploads the data tables and publishes the geoscience object. Displaying the returned object renders a rich HTML summary with links to the Evo Portal and Viewer.

In [ ]:
from evo.objects.typed.downhole_collection import DownholeCollection

dhc = await DownholeCollection.create(manager, dhc_data)
dhc